In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/workdata'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/workdata/fake/id16_id3_0003/frame_0.jpg
/kaggle/input/workdata/fake/id16_id3_0003/frame_1.jpg
/kaggle/input/workdata/fake/id48_id41_0003/frame_0.jpg
/kaggle/input/workdata/fake/id48_id41_0003/frame_1.jpg
/kaggle/input/workdata/fake/id27_id28_0008/frame_0.jpg
/kaggle/input/workdata/fake/id27_id28_0008/frame_1.jpg
/kaggle/input/workdata/fake/id41_id45_0001/frame_0.jpg
/kaggle/input/workdata/fake/id41_id45_0001/frame_1.jpg
/kaggle/input/workdata/fake/id53_id50_0005/frame_0.jpg
/kaggle/input/workdata/fake/id53_id50_0005/frame_1.jpg
/kaggle/input/workdata/fake/id53_id50_0005/frame_2.jpg
/kaggle/input/workdata/fake/id38_id30_0002/frame_0.jpg
/kaggle/input/workdata/fake/id38_id30_0002/frame_1.jpg
/kaggle/input/workdata/fake/id3_id0_0000/frame_0.jpg
/kaggle/input/workdata/fake/id3_id0_0000/frame_1.jpg
/kaggle/input/workdata/fake/id3_id0_0000/frame_2.jpg
/kaggle/input/workdata/fake/id45_id48_0009/frame_0.jpg
/kaggle/input/workdata/fake/id45_id48_0009/frame_1.jpg
/kaggle/input/work

In [ ]:
# 1. GPU SETUP
try:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Configured {len(gpus)} GPUs.")
    
    # Enable Mixed Precision
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    
except RuntimeError:
    pass # Already initialized

Configured 2 GPUs.


In [5]:
# System & Helper Libraries
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils import shuffle

# Image Processing 
import cv2
from PIL import Image, ImageEnhance

# TensorFlow & Keras Core
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras import Input

#  Keras Layers 
from tensorflow.keras.layers import (
    Conv2D, Dense, Flatten, GlobalAveragePooling1D, 
    Reshape, Add, LayerNormalization, MultiHeadAttention, 
    Lambda, Dropout, BatchNormalization, Activation
)

# Optimizers & Preprocessing
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Pre-trained Models 
from tensorflow.keras.applications import VGG16          
from tensorflow.keras.applications import MobileNetV3Small,ResNet50

# Configuration
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(f"GPU Detected: {physical_devices[0].name}")
else:
    print("No GPU detected.")
print(f"TensorFlow Version: {tf.__version__}")

GPU Detected: /physical_device:GPU:0
TensorFlow Version: 2.19.0


In [3]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, applications, mixed_precision
from glob import glob
from sklearn.utils import shuffle

2026-02-07 09:36:13.490156: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770456973.511349     231 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770456973.517836     231 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770456973.534957     231 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770456973.534983     231 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770456973.534985     231 computation_placer.cc:177] computation placer alr

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import models
from mtcnn import MTCNN
import pandas as pd
import time

# 1. CONFIGURATION
# CHANGED: Pointing to the Real videos folder
VIDEO_FOLDER = "/kaggle/input/celeb-df-v2/YouTube-real"
MODEL_PATH = "/kaggle/input/deepfake/tensorflow2/default/1/deepfake_model (1).h5"

SEQ_LENGTH = 10
IMG_SIZE = 224

# 2. LOAD MODEL
print(f"Loading model from {MODEL_PATH}...")
model = models.load_model(MODEL_PATH)
print("Model loaded successfully!")

# Initialize Face Detector
detector = MTCNN()

# 3. VIDEO PROCESSOR
def process_video(video_path):
    if not os.path.exists(video_path): return None
    
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    
    if not frames: return None

    # Face Extraction
    face_crops = []
    stride = max(1, len(frames) // SEQ_LENGTH)
    
    for i in range(0, len(frames), stride):
        if len(face_crops) >= SEQ_LENGTH: break
        
        frame = frames[i]
        results = detector.detect_faces(frame)
        
        if results:
            x, y, w, h = results[0]['box']
            x, y = max(0, x), max(0, y)
            face = frame[y:y+h, x:x+w]
            
            try:
                face = cv2.resize(face, (IMG_SIZE, IMG_SIZE)) / 255.0
                face_crops.append(face)
            except: pass

    if not face_crops: return None

    while len(face_crops) < SEQ_LENGTH:
        face_crops.extend(face_crops)
        
    return np.array([face_crops[:SEQ_LENGTH]])

# 4. BATCH TEST RUNNER 
def test_youtube_real(limit=None):
    print(f"\nStarting Evaluation on: {VIDEO_FOLDER}")
    
    all_videos = [f for f in os.listdir(VIDEO_FOLDER) if f.endswith('.mp4')]
    
    if limit:
        all_videos = all_videos[:limit]
        
    total = len(all_videos)
    print(f"Processing {total} videos...")
    
    results = []
    real_detected_count = 0  # Counter for CORRECT Real predictions
    start_time = time.time()
    
    for i, filename in enumerate(all_videos):
        video_path = os.path.join(VIDEO_FOLDER, filename)
        
        try:
            input_data = process_video(video_path)
            
            if input_data is None:
                res = "ERROR (No Face)"
                conf = 0.0
            else:
                pred = model.predict(input_data, verbose=0)[0][0]
                
                # LOGIC FLIPPED FOR REAL VIDEOS
                # Prediction >= 0.5 means FAKE (which is WRONG for this dataset)
                # Prediction < 0.5 means REAL (which is CORRECT for this dataset)
                
                if pred >= 0.5:
                    res = "FAKE (False Positive)"
                    conf = pred
                else:
                    res = "REAL"  # This is what we want!
                    conf = 1 - pred
                    real_detected_count += 1 # Count success
            
            # Print update every 10 videos
            if (i+1) % 10 == 0:
                print(f"[{i+1}/{total}] Last: {filename} -> {res} ({conf:.2f})")
                
            results.append({"video": filename, "result": res, "confidence": conf})
            
        except Exception as e:
            print(f"Error on {filename}: {e}")

    # SUMMARY 
    elapsed = time.time() - start_time
    df = pd.DataFrame(results)
    
    valid_predictions = df[df['result'] != "ERROR (No Face)"]
    
    if len(valid_predictions) > 0:
        # Calculate accuracy based on how many were correctly identified as REAL
        correct_reals = len(valid_predictions[valid_predictions['result'] == "REAL"])
        accuracy = (correct_reals / len(valid_predictions)) * 100
    else:
        accuracy = 0.0
        
    print("\n" + "="*40)
    print(f"DONE in {elapsed/60:.2f} min")
    print(f"ACCURACY ON REALS: {accuracy:.2f}%") 
    print(f"   - Correctly Detected as Real: {real_detected_count}")
    print(f"   - Incorrectly Labeled as Fake: {len(valid_predictions) - real_detected_count}")
    print("="*40)
    
    # Save to a new CSV file
    df.to_csv("youtube_real_results.csv", index=False)

# Run (YouTube-real has about ~590 videos, so limit=1000 covers all of them)
test_youtube_real(limit=1000)

Loading model from /kaggle/input/deepfake/tensorflow2/default/1/deepfake_model (1).h5...


I0000 00:00:1770456988.049938     231 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1770456988.055068     231 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model loaded successfully!

Starting Evaluation on: /kaggle/input/celeb-df-v2/YouTube-real
Processing 300 videos...


I0000 00:00:1770457010.921484     295 cuda_dnn.cc:529] Loaded cuDNN version 91002


[10/300] Last: 00012.mp4 -> REAL (1.00)
[20/300] Last: 00249.mp4 -> REAL (1.00)
[30/300] Last: 00262.mp4 -> REAL (1.00)
[40/300] Last: 00237.mp4 -> FAKE (False Positive) (0.78)
[50/300] Last: 00060.mp4 -> REAL (1.00)
[60/300] Last: 00127.mp4 -> REAL (1.00)
[70/300] Last: 00008.mp4 -> FAKE (False Positive) (0.73)
[80/300] Last: 00122.mp4 -> REAL (1.00)
[90/300] Last: 00097.mp4 -> REAL (1.00)
[100/300] Last: 00020.mp4 -> REAL (1.00)
[110/300] Last: 00080.mp4 -> REAL (1.00)
[120/300] Last: 00165.mp4 -> FAKE (False Positive) (0.84)
[130/300] Last: 00285.mp4 -> REAL (1.00)
[140/300] Last: 00212.mp4 -> REAL (1.00)
[150/300] Last: 00102.mp4 -> REAL (1.00)
[160/300] Last: 00251.mp4 -> FAKE (False Positive) (0.99)
[170/300] Last: 00054.mp4 -> REAL (1.00)
[180/300] Last: 00005.mp4 -> REAL (1.00)
[190/300] Last: 00145.mp4 -> REAL (1.00)
[200/300] Last: 00098.mp4 -> REAL (1.00)
[210/300] Last: 00113.mp4 -> REAL (1.00)
[220/300] Last: 00264.mp4 -> REAL (1.00)
[230/300] Last: 00091.mp4 -> REAL (1.00

In [ ]:
!pip install lz4

In [ ]:
pip install MTCNN

In [ ]:
pip install lz4